In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoModelForCausalLM
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')

In [ ]:
model_id = 'google/mt5-base'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

In [ ]:
dataset_id = 'vohuutridung/3190-data'
train_ds = load_dataset(dataset_id, split='train')
valid_ds = load_dataset(dataset_id, split='validation')

In [ ]:
MAX_INPUT_LEN = 256
MAX_OUTPUT_LEN = 256

def preprocess(example):
    model_inputs = tokenizer(
        example["text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            str(example["labels"]),
            max_length=MAX_OUTPUT_LEN,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds.map(preprocess, batched=False)
valid_ds = valid_ds.map(preprocess, batched=False)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5base_absa",
    save_total_limit=1,    # <--- QUAN TRỌNG NHẤT: Chỉ giữ lại đúng 1 file checkpoint cuối cùng
    save_steps=600, 
    save_only_model=True,
    # <--- Tăng lên: Đừng lưu quá thường xuyên (ví dụ 500 bước mới lưu 1 lần)
    per_device_train_batch_size=2,  # <--- GIẢM XUỐNG 4 (hoặc 8 nếu 4 vẫn dư)
    per_device_eval_batch_size=8,   # <--- GIẢM XUỐNG 4
    gradient_accumulation_steps=8,  # <--- THÊM DÒNG NÀY (để tổng batch vẫn là 4*4=16)
    num_train_epochs=5,
    fp16=True,
    eval_steps=200,
    eval_strategy="steps",
    #save_steps=200,
    #save_total_limit=1,
    save_strategy="steps",
    logging_steps=100,
    
    report_to="tensorboard",
    predict_with_generate=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
import shutil
import os

path = "./mt5base_absa" # Tên thư mục output_dir của bạn
if os.path.exists(path):
    shutil.rmtree(path)
    print(f"Đã xóa sạch thư mục {path} để giải phóng ổ cứng!")
else:
    print("Thư mục không tồn tại, ổ cứng có thể đã sạch.")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)


REPO_ID = 'philong68/mt5-base-absa-v2'
trainer.model.push_to_hub(REPO_ID)
trainer.processing_class.push_to_hub(REPO_ID)

In [ ]:
def infer(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(infer("Pin máy dùng được lâu"))
